<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Building a Physics-Informed Neural Network for real-world applications**


Estimated time needed: **45** minutes


In the age of AI, neural networks solve complex problems, but **don't forget the real world—physical principles!** Integrating these physics principles into machine learning ensures precise understanding.

In this lab, you'll learn to build and train a **Physics-Informed Neural Network (PINN) in PyTorch** by solving a **1D thermal diffusion equation**. You'll compare your PINN results with a traditional numerical solution, demonstrating how physics-informed AI enhances accuracy in modeling real-world phenomena.


## __Table of contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
            <li><a href="#Device-Check-for-PyTorch">Device Check for PyTorch</a></li>
        </ol>
    </li>
    <li>
        <a href="#Background">Background</a>
        <ol>
            <li><a href="#Physics-Informed-Neural-Networks-(PINNs)-Theory">Physics-Informed Neural Networks (PINNs) Theory</a></li>
            <li><a href="#Thermal/Chemical-Diffusion-Theory-&amp;-Numerical-Solution">Thermal/Chemical Diffusion Theory & Numerical Solution</a></li>
            <li><a href="#Why-PINNs?">Why PINNs?</a></li>
        </ol>
    </li>
    <li>
        <a href="#The-Model">The Model</a>
        <ol>
            <li><a href="#The-Neural-Network">The Neural Network</a></li>
            <li><a href="#The-Loss-Function">The Loss Function</a></li>
            <li><a href="#Training">Training</a></li>
        </ol>
    </li>
    <li>
        <a href="#Result">Result</a>
        <ol>
            <li><a href="#Comparing-to-Theoretical-Solution">Comparing to Theoretical Solution</a></li>
            <li><a href="#Visualizing-Training-Convergence">Visualizing Training Convergence</a></li>
        </ol>
    </li>
</ol>



## Objectives

After completing this lab, you are able to:

- Understand the theory behind Physics-Informed Neural Networks (PINNs).
- Build and implement a PINN using PyTorch.
- Solve partial differential equations (PDEs) using PINNs.


----


## Setup


For this lab, you use the following libraries:

*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.
*   [`PyTorch`](https://pytorch.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and machine-learning-pipeline related functions.
*   [`scipy`](https://seaborn.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) to calculate the solution in traditional numerical method.
*   [`matplotlib`](https://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for additional plotting tools.


### Installing required libraries

Before we dive into building our Physics-Informed Neural Network (PINN), we need to ensure that all necessary libraries are installed. This step will set up the environment required for the lab, including `PyTorch` and any other essential packages.


In [ ]:
%pip install torch numpy matplotlib scipy

Run the above command to install the required libraries. **The installation might take ~ 5 minutes**, please be patient.

After running the above command, **restart the kernel** to ensure the libraries are installed correctly. This will help avoid any issues when we start coding.

Once the installation is complete, we'll proceed to the next step.


### Importing required libraries

_We recommend you import all required libraries in one place (here):_


In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from scipy.special import erf

# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Device check for PyTorch

`PyTorch` could be accelerated by GPU. If you have GPU in your local device, you can download this notebook and use GPU locally.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Background


### Physics-Informed Neural Networks (PINNs) Theory

Physics-Informed Neural Networks (PINNs) are an innovative approach that blends traditional neural networks with physical principles to solve complex problems. 

[Paper for PINN](https://www.sciencedirect.com/science/article/abs/pii/S0021999118307125) is published in *Journal of Computational Physics* in 2019.

Unlike conventional neural networks that rely solely on data, **PINNs incorporate the underlying physical laws, such as conservation laws, boundary conditions, and differential equations, directly into the learning process**. This integration allows the model to be more accurate, reliable, and interpretable, especially when data is scarce or noisy.

In a PINN, the **loss function is augmented with terms that represent the residuals of the governing differential equations**. By minimizing this loss function, the neural network not only fits the data but also satisfies the physical laws. This makes PINNs particularly powerful for solving problems where the governing equations are well-known, such as fluid dynamics, heat transfer, and material deformation.


### Thermal/Chemical Diffusion Theory

The 1D diffusion equation is a fundamental partial differential equation (PDE) that describes how a quantity such as heat, particles, or other substances diffuses through a medium over time. The equation is expressed as:

$$
\frac{\partial u(x,t)}{\partial t} = D \frac{\partial^2 u(x,t)}{\partial x^2}
$$
 
Where:
- $\partial u(x,t)$ is the quantity of interest (e.g., temperature or concentration) at position $x$ and time $t$.
- $D$ is the diffusion coefficient, which determines the rate of diffusion.

In this lab, we focus on solving the 1D diffusion equation using PINN and compare it with the mathematical solution in the format of [error function](https://en.wikipedia.org/wiki/Error_function). 

For the other traditional methods of solving general PDEs, typically involves discretizing the equation using [finite difference methods (FDMs)](https://en.wikipedia.org/wiki/Finite_difference_method), where the continuous domain is broken down into discrete points, and the PDE is approximated using difference equations.


## Why PINNs?

While traditional numerical methods are effective for solving PDEs, they can become computationally expensive, **especially for high-dimensional problems or complex geometries**. Additionally, these methods may require extensive grid refinement and stabilization techniques to achieve accurate solutions.

Physics-Informed Neural Networks offer a powerful alternative by embedding the physical laws directly into the neural network architecture. **PINNs can efficiently handle problems with complex boundary conditions and irregular domains without the need for extensive meshing or discretization**. Moreover, they can be **trained using a relatively small amount of data**, as the physical laws guide the learning process, ensuring that the solution is both physically consistent and data-driven.

By using PINNs to solve the 1D diffusion equation, you will gain insight into how this approach can offer advantages over traditional methods, particularly in terms of flexibility, scalability, and the ability to generalize to more complex scenarios.


## The Model

### The Neural Network

The core of our Physics-Informed Neural Network (PINN) is a fully connected feed-forward neural network that maps the input space (spatial and temporal coordinates) to the output space (solution of the differential equation). 

In this implementation, the `PINNDiffusion` class defines the architecture of our neural network. It consists of several layers:

- The network takes two inputs: the spatial coordinate `x` and the temporal coordinate `t`, which are concatenated together to form the input to the network.
- The network has four hidden layers, each with 50 neurons and Tanh activation functions. These hidden layers capture the complex interactions between space and time.
- The final output layer has a single neuron, representing the predicted solution \( $u(x,t)$ \).

This design allows the network to approximate the solution to the 1D diffusion equation effectively. You can adjust the Neural Network and activation functions based on the complexity of actual tasks.


In [ ]:
class PINNDiffusion(nn.Module):
    def __init__(self):
        super(PINNDiffusion, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 1)
        )

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

### The Loss Function

**Using $D$ diffusion coefficient 0.01 as an example.**

The loss function is critical in training a PINN, as it enforces both the data-driven component and the physical constraints derived from the governing differential equation. The `pinn_loss` function computes the loss based on three components:

- **PDE Loss**: This is the primary loss component, ensuring that the network's output satisfies the 1D diffusion equation. It is calculated by taking the time derivative \( $u_t$ \) and the second spatial derivative \( $u_{xx}$ \) of the predicted solution. The PDE residual is computed as \( $u_t - D \times u_{xx}$ \), $D = 0.01$ and the loss is the mean squared error of this residual.
  
- **Initial Condition Loss**: This term ensures that the network's output satisfies the initial condition of the diffusion equation. **Here, we assume that the initial condition is a step function**, where \( $u(x, t=0)$ \) is 0 for \( $x \leq 0.5$ \) and 1 for \( $x > 0.5$ \).

- **Boundary Condition Loss**: This term ensures that the network's output satisfies the boundary conditions. Specifically, it penalizes deviations from \( $u(x=0, t) = 0$ \) and \( $u(x=1, t) = 1$ \).

The final loss is the sum of these three components, encouraging the network to produce solutions that are consistent with the physical laws and boundary/initial conditions.


In [ ]:
def pinn_loss(x, t, u):
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    
    pde_loss = u_t - 0.01 * u_xx
    
    # Loss for initial condition and boundary condition
    ic_loss = torch.mean((u[t.squeeze() == 0] - (x[t.squeeze() == 0] > 0.5).float())**2)
    bc_loss = torch.mean(u[x.squeeze() == 0]**2) + torch.mean((u[x.squeeze() == 1] - 1)**2)
    
    return torch.mean(pde_loss**2) + ic_loss + bc_loss

### Training

The training process involves iteratively optimizing the neural network to minimize the loss function, thereby improving the accuracy of the predicted solution. The `train` function outlines this process:

- **Data Preparation**: The spatial and temporal coordinates are generated over a grid using `torch.linspace` and then reshaped into the required format for input into the neural network.

- **Forward Pass**: For each training iteration, the network predicts the solution \( $u(x,t)$ \) for the given spatial and temporal coordinates.

- **Loss Calculation**: The `pinn_loss` function is used to compute the loss based on the predicted solution, enforcing the PDE, initial conditions, and boundary conditions.

- **Backward Pass and Optimization**: The loss is backpropagated through the network to compute the gradients, and the optimizer updates the network's weights to minimize the loss.

*In the code, ```loss.backward()``` and ```optimizer.step()``` set up the backward pass and optimization.*

- **Saving Model States**: During training, the model’s state is saved for later analysis or use.


In [ ]:
def train(model, optimizer, epochs):
    saved_models = {}
    epoch_saved = []
    for epoch in tqdm(range(epochs), desc="Training Progress"):
        optimizer.zero_grad()
        
        x = torch.linspace(0, 1, 100, requires_grad=True).unsqueeze(1).to(device)
        t = torch.linspace(0, 1, 100, requires_grad=True).unsqueeze(1).to(device)
        x, t = torch.meshgrid(x.squeeze(), t.squeeze())
        x = x.reshape(-1, 1)
        t = t.reshape(-1, 1)
        
        u = model(x, t)
        loss = pinn_loss(x, t, u)
        
        loss.backward()
        optimizer.step()
        
         # Save the mode
        saved_models[epoch+1] = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epoch_saved.append(epoch + 1)

    return model, epoch_saved, saved_models

You can set up and train the model. This model is trained over 1500 epochs, and the loss is monitored. If you have GPU locally, you can train the model faster with more epochs for better result.

This training process allows the PINN to learn a solution that not only fits the data but also adheres to the underlying physical principles of the diffusion equation.


In [ ]:
model = PINNDiffusion().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the model
model, epoch_saved, saved_models = train(model, optimizer, epochs=1500)

The training will take **~10 minutes** for the 1500 epochs, be patient and have a cup of coffee.


## Result

### Comparing to Theoretical Solution


Here we have the trained PINN for the 1D diffusion task after a 1000 epoch run. We'd like to visualize the result. Before that, we need the theoretical mathematical solution for the diffusion, which can be expressed as:

$$u(x,t) = \frac{1}{2}\left( 1 + \text{erf}\left(\frac{x - 0.5}{2\sqrt{Dt}}\right)\right)$$

Where:

-  $u(x,t)$  is the solution at position $x$ and time $t$
-  $\text{erf}$  is the error function
-  $D$  is the diffusion coefficient (set to 0.01 in this code)

Now, we can compare the results from the theoretical solution and the PINN model for $t = 0.25, 0.5, 0.75, 1$.


In [ ]:
def plot_all_solutions(model):
    model.eval()
    with torch.no_grad():
        x = torch.linspace(0, 1, 100).unsqueeze(1).to(device)
        t_values = [0.25, 0.5, 0.75, 1.0]
        
        plt.figure(figsize=(15, 10))
        
        # Plot initial condition
        plt.plot(x.cpu().numpy(), np.heaviside(x.cpu().numpy() - 0.5, 0.5), 'k--', label='Initial condition')
        
        # Plot PINN solutions
        for t in t_values:
            t_tensor = torch.full_like(x, t).to(device)
            u = model(x, t_tensor).cpu().numpy()
            plt.plot(x.cpu().numpy(), u, label=f'PINN t={t:.2f}')
        
        # Plot theoretical solutions
        D = 0.01  # Diffusion coefficient
        x_np = x.cpu().numpy()
        for t in t_values:
            u_theoretical = 0.5 * (1 + erf((x_np - 0.5) / (2 * np.sqrt(D * t))))
            plt.plot(x_np, u_theoretical, '--', label=f'Theoretical t={t:.2f}')
        
        plt.title('PINN vs Theoretical Diffusion Solutions')
        plt.xlabel('x')
        plt.ylabel('u(x,t)')
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
plot_all_solutions(model)

We can see that despite some minor differences, the PINN successfully modeled the result for the 1D thermal/chemical diffusion.


### Visualizing Training Convergence


Let's visualize how the training converges over the epochs, using $t=1$ as an example.


In [ ]:
def plot_epoch_comparison(epoch_saved, saved_models, device):
    fig, ax = plt.subplots(figsize=(7, 5), dpi=100)
    x = torch.linspace(0, 1, 100).unsqueeze(1).to(device)
    t = torch.full_like(x, 1.0).to(device)
    
    # Theoretical solution at t=1
    D = 0.01
    x_np = x.cpu().numpy()
    u_theoretical = 0.5 * (1 + erf((x_np - 0.5) / (2 * np.sqrt(D))))
    
    ax.plot(x_np, u_theoretical, 'k--', label='Theoretical t=1.0')
    line, = ax.plot([], [], 'r-', label='PINN Solution')
    
    ax.set_title('PINN Solutions at t=1.0 for Different Epochs')
    ax.set_xlabel('x')
    ax.set_ylabel('u(x,1)')
    ax.legend()
    ax.grid(True)
    
    def init():
        line.set_data([], [])
        return line,
    
    def update(epoch):
        temp_model = PINNDiffusion().to(device)
        temp_model.load_state_dict(saved_models[epoch])
        temp_model.eval()
        with torch.no_grad():
            u = temp_model(x, t).cpu().numpy()
        line.set_data(x_np, u)
        ax.set_title(f'PINN Solution at t=1.0 for Epoch {epoch}')
        return line,
    
    ani = animation.FuncAnimation(fig, update, frames=epoch_saved, init_func=init, blit=True, interval=500)
    
    plt.close(fig)  # Prevents static plot from showing up in notebooks
    
    return ani

In [ ]:
epoch_show = my_list = [1] + list(range(10, 651, 10)) + list(range(650, 1501, 50))
anim = plot_epoch_comparison(epoch_show, saved_models, device)

In [ ]:
# Usage example
from IPython.display import HTML
HTML(anim.to_jshtml())

## Conclusion

Congratulation!

Now that you've successfully used a Physics-Informed Neural Network (PINN) to solve a partial differential equation (PDE), you've taken the first step in applying machine learning to physics-based problems. This project provided a simple example using a 1D diffusion PDE, but the principles you've learned can be **extended to much more complex scenarios**.

For more advanced and intricate work, you can **adjust the neural network architecture and modify the loss function to tackle higher-dimensional problems, more complex boundary conditions, or different types of PDEs**. These adjustments can enable the PINN to solve a wide range of physics-based problems, from fluid dynamics to material science.

If you're interested in diving deeper into **more advanced Physics-Informed Machine Learning projects**, consider exploring the following paper list in my [github repository](https://github.com/JCJunxing/Paper_summary_PDE). These works highlight cutting-edge approaches, including the use of Transformer models and other advanced methods like Fourier Neural Operators (FNOs), here are some examples:

| No. | Title | Link |
|-----|-------|------|
| 1 | Physics informed token transformer for solving partial differential equations | [link](https://iopscience.iop.org/article/10.1088/2632-2153/ad27e3/pdf) |
| 2 | Multi-scale time-stepping of Partial Differential Equations with transformers | [link](https://www.sciencedirect.com/science/article/pii/S0045782524002391?ref=pdf_download&fr=RR-2&rr=8b2624013d22a229) |
| 3 | Transformer for Partial Differential Equations’ Operator Learning | [link](https://openreview.net/pdf?id=EPPqt3uERT) |
| 4 | Fourier Neural Operator for Parametric Partial Differential Equations | [link](https://openreview.net/pdf?id=c8P9NQVtmnO) |
| 5 | Learning the Solution Operator of Boundary Value Problems using Graph Neural Networks | [link](https://arxiv.org/pdf/2206.14092) |

These resources will guide you through more sophisticated applications and show how state-of-the-art machine learning models, like Transformers and Fourier Neural Operators (FNOs), are being used to push the boundaries of Physics-Informed Machine Learning.

Feel free to explore these references and apply the techniques you’ve learned to new and challenging problems in your domain of interest!


## Authors


[J.C.(Junxing) Chen](https://www.linkedin.com/in/jcjunxing/) Research Scientist


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-08-15|0.1|J.C.(Junxing) Chen|Create lab|


Copyright © 2024 IBM Corporation. All rights reserved.
